# Method comparison: WAPER vs Zimin envelope — April 2011

This notebook visualises the sweep results produced by `scripts/method_comparison/run_sweep.py`.
Run that module first (~2–3 h) to populate `results/method_comparison_sweep.csv`.

```bash
mkdir -p results
python -m scripts.method_comparison.run_sweep
```

Then execute all cells below.

In [ ]:
# Bootstrap: anchor to the repo root so relative paths and `scripts.` imports
# work no matter which directory Jupyter started the kernel in. Jupyter sets the
# kernel CWD to the notebook's own folder, so we walk up to the repo root (the
# directory containing 'datasets/' and 'scripts/') and put it on sys.path.
import os, sys, pathlib

root = pathlib.Path.cwd()
while not (root / "datasets").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root:", os.getcwd())

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
df = pd.read_csv("results/method_comparison_sweep.csv")
fig, ax = plt.subplots(figsize=(7, 4))
for method, g in df.groupby("method"):
    g = g.sort_values("threshold")
    ax.plot(g["threshold"], g["mean_iou"], marker="o", label=method)
    best = g.loc[g["mean_iou"].idxmax()]
    ax.scatter([best["threshold"]], [best["mean_iou"]], s=120,
               facecolors="none", edgecolors="k", zorder=5)
ax.set_xlabel("threshold (GT in (m/s)/km, or ST in m/s)")
ax.set_ylabel("mean IoU vs Zimin (14 m/s)")
ax.set_title("Agreement with Zimin envelope — April 2011")
ax.legend(); fig.tight_layout()

In [ ]:
import cartopy.crs as ccrs
from scripts.method_comparison.run_sweep import load_dataset, compute_zimin_masks, run_base_waper
from scripts.method_comparison.masks import band_mask, pixel_lonlat_grid, edge_pruning_mask

v = load_dataset()
band = band_mask(20.0, 80.0)
zimin = compute_zimin_masks(v, band)
plon, plat = pixel_lonlat_grid("north")

# best thresholds from the CSV
best_gt = df[df.method == "edge_pruning"].sort_values("mean_iou").iloc[-1]["threshold"]
w = run_base_waper(v, node_pruning_threshold=20, edge_pruning_threshold=float(best_gt))
method_masks = np.stack([edge_pruning_mask(tsd, band) for tsd in w._time_step_data])

agree = (method_masks & zimin).mean(axis=0)
method_only = (method_masks & ~zimin).mean(axis=0)
zimin_only = (~method_masks & zimin).mean(axis=0)

# shared vmax so all three panels are on the same frequency scale
vmax = max(agree.max(), method_only.max(), zimin_only.max())

fig, axes = plt.subplots(1, 3, figsize=(15, 5),
                         subplot_kw={"projection": ccrs.NorthPolarStereo()})
for ax, fld, title in zip(axes, [agree, method_only, zimin_only],
                          ["agree", "edge-only", "zimin-only"]):
    ax.coastlines(); ax.set_extent([-180, 180, 20, 80], ccrs.PlateCarree())
    im = ax.pcolormesh(plon, plat, np.where(band, fld, np.nan),
                       transform=ccrs.PlateCarree(), vmin=0, vmax=vmax)
    ax.set_title(title)
fig.colorbar(im, ax=axes.tolist(), label="frequency", shrink=0.7)
fig.tight_layout()

In [ ]:
t = 24 * 14  # 2011-04-15 00Z (index into hourly series)
fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={"projection": ccrs.NorthPolarStereo()})
ax.coastlines(); ax.set_extent([-180, 180, 20, 80], ccrs.PlateCarree())
cf = ax.contourf(v.longitude, v.latitude, v.isel(time=t),
                 levels=np.linspace(-40, 40, 17), cmap="RdBu_r",
                 transform=ccrs.PlateCarree())
fig.colorbar(cf, ax=ax, label="v (m/s)", shrink=0.7)
for mask, color in [(zimin[t], "k"), (method_masks[t], "lime")]:
    ax.contour(plon, plat, mask.astype(float), levels=[0.5],
               colors=color, transform=ccrs.PlateCarree())
ax.set_title("v field + Zimin (black) vs edge-pruning (green), 2011-04-15 00Z")
fig.tight_layout()